# Sample Preparation of Specification File
### This Jupyter Notebooks aids in the preparation of a Specification File, to serve as a blueprint for the generation of the Define-XML file. \ The source of this code include final SDTM datasets, annotations as generated within the previous tutorial and libraries to be called from "sdtm_34_dictionaries.py"
\
Importing key modules:

In [23]:
import os
import shutil
import pandas as pd
import numpy as np
import pyreadstat as prs
from openpyxl import load_workbook

Importing dictionaries:

In [24]:
import sdtm_34_dictionaries

Set standard variables and format for the dictionary of supplemental qualifiers:

In [25]:
df_supp = pd.DataFrame(sdtm_34_dictionaries.dict_supp).T
df_supp = df_supp.fillna({'Origin':'Collected'})
df_supp.reset_index(inplace=True)
df_supp['Variable']=df_supp['index']
df_supp

,index,Origin,Source,Core,Variable Type,Variable Order,Variable
0,STUDYID,Protocol,Sponsor,Req,SDTM,1,STUDYID
1,RDOMAIN,Derived,Sponsor,Req,SDTM,2,RDOMAIN
2,USUBJID,Derived,Sponsor,Req,SDTM,3,USUBJID
3,IDVAR,Assigned,Sponsor,Exp,SDTM,4,IDVAR
4,IDVARVAL,Derived,Sponsor,Exp,SDTM,5,IDVARVAL
5,QNAM,Assigned,Sponsor,Req,SDTM,6,QNAM
6,QLABEL,Assigned,Sponsor,Req,SDTM,7,QLABEL
7,QVAL,Collected,Investigator,Req,SDTM,8,QVAL
8,QORIG,Assigned,Sponsor,Req,SDTM,9,QORIG
9,QEVAL,Assigned,Sponsor,Exp,SDTM,10,QEVAL


Setting path to folder where annotations are saved and to folder where specification file must be saved:

In [26]:
annotation_path = "Output"
specification_path = "Specification"

Setting the study, Clinical Study A is identified as "0224"; Clinical Study B is identified as "0218":

In [28]:
study='0218'

Setting the template specification file to be used, and where to save metadata into:

In [34]:
template_spec = "Specification/template_spec_sdtm.xlsx"
output_spec = "Specification/spec_sdtm_"+study+".xlsx"
book = load_workbook(template_spec)

Just on the first run, set the toggle "copy" to 1, so that it creates a new copy of the template specification file to the desired location, after that, the toggle can be set to 0, so that new modules can be appended to the excel file at different times, without restarting from scratch:

In [36]:
copy=0

In [ ]:
if copy==1:
    shutil.copyfile(template_spec, output_spec)

Setting SDTM folder for pyreadstat:

In [37]:
folder_sdtm = "Sample Datasets"

Set mapping for data type when reading SAS datasets:

In [38]:
mapping = {'string':'Char','double':'Num'}

### Sample Metadata Collection for Demographics Dataset (DM) and Supplemental Qualifier Variables for DM (SUPPDM)
First, annotation files are recollected, based on the mapping:

In [40]:
dmtot = pd.read_csv(os.path.join(annotation_path,"dm_annotation_0218.csv"),sep=';')
dmtot = dmtot[~dmtot['pred'].str.startswith('RP')]
dmtot['original_module']='DM'
suppie = pd.read_csv(os.path.join(annotation_path,"suppie_annotation_0218.csv"),sep=';')
suppie = suppie[suppie['pred']=='DMCOHORT']
suppie['original_module']='IE'
suppdm = pd.read_csv(os.path.join(annotation_path,"suppdm_annotation_0218.csv"),sep=';')
suppdm = suppdm[suppdm['pred'].isin(['DMCOHORT','RACEOTH'])]
suppdm['original_module']='DM'
dm = pd.concat([dmtot,suppie,suppdm])
dm

,Unnamed: 0,0,pred,original_module
0,0,STUDY,STUDYID,DM
1,1,MODULE,DOMAIN,DM
2,3,PATNO,SUBJID,DM
3,7,DSSTMO_N,RFICDTC2,DM
4,8,DMMO_N,DMDTC2,DM
5,9,BRTHMO_N,BRTHDTC2,DM
6,10,_CLIENT_TIMESTAMP,DMDTC,DM
7,15,SEX,SEX,DM
8,17,AGE,AGE,DM
9,18,BRTHDY,BRTHDTC1,DM


In [41]:
dm['Variable Name'] = dm['pred']
dm['Conversion'] = dm['original_module'] + "." + dm['0']
dm.drop(['0','Unnamed: 0','pred'], axis=1, inplace=True)
#cm['Origin']='Collected'
dm['Variable']=dm['Variable Name'].str.replace(r'\d+', '', regex=True)
dm['numbers']=dm['Variable Name'].str.replace(r'^[A-Z]*(?!\d){1}$', '', regex=True)
dm2 = dm.groupby('Variable')['Conversion'].agg(', '.join).reset_index()
dm = pd.merge(dm[['Variable Name','Variable']],dm2,on='Variable')
#cm['Conversion Definition'] = np.where(cm['numbers']!="",cm['numbers'],cm['Conversion'])
dm['Conversion Definition']=dm['Conversion']
dm.drop(['Conversion','Variable Name'], axis=1, inplace=True)
dm.drop_duplicates('Variable', inplace=True)
dm

,Variable,Conversion Definition
0,STUDYID,DM.STUDY
1,DOMAIN,DM.MODULE
2,SUBJID,DM.PATNO
3,RFICDTC,"DM.DSSTMO_N, DM.DSSTDY, DM.DSSTYR"
4,DMDTC,"DM.DMMO_N, DM._CLIENT_TIMESTAMP, DM.DMDY, DM.DMYR"
5,BRTHDTC,"DM.BRTHMO_N, DM.BRTHDY, DM.BRTHYR"
7,SEX,DM.SEX
8,AGE,DM.AGE
15,RACE,DM.RACE
16,ETHNIC,DM.ETHNIC


Now, SDTM Datasets are collected in the folder where they are stored. Please consider, that some fields have been masked in the columns (such as "DMCOHORT", "DTHDTC", "DTHFL") from the original datasets, although this is not preventing metadata from its collection.

In [42]:
ae001, ae001_meta = prs.read_sas7bdat(folder_sdtm+"/dm.sas7bdat")
column_info = pd.DataFrame()
column_info['Variable'] = ae001_meta.column_names
column_info.set_index('Variable',inplace=True)
column_info['Member'] = "SDTM.DM"
column_info['Type'] = ae001_meta.readstat_variable_types
column_info['Type'] = column_info['Type'].map(mapping)
column_info['Len'] = ae001_meta.variable_storage_width
column_info['Label'] = ae001_meta.column_labels
column_info['Format'] = ae001_meta.value_labels
column_info.reset_index(inplace=True)

ae002, ae002_meta = prs.read_sas7bdat(folder_sdtm+"/suppdm.sas7bdat")
unique_supp = list(ae002.QNAM.unique())
ae002['Len'] = ae002['QVAL'].str.len()
ae002.sort_values('Len', inplace=True)
ae002.drop_duplicates('QNAM', keep='first', inplace=True)
len_df = ae002[['QNAM','Len']].set_index('QNAM')
lengths = len_df.to_dict()['Len']
labels_df = ae002[['QNAM','QLABEL']].set_index('QNAM')
labels = labels_df.to_dict()['QLABEL']
d = {}
for u in unique_supp:
    d[unique_supp.index(u)] = u
    
d_df = pd.DataFrame(d,index=d.values())
d_df['Type'] = 'Char'
d_df['Len'] = lengths
d_df['Label'] = labels
d_df = d_df[['Type', 'Len','Label']].reset_index()
d_df['Variable'] = d_df['index']
d_df.drop('index', axis=1, inplace=True)
d_df['Member'] = "SDTM.SUPPDM"

column_info_tot = pd.concat([column_info,d_df])
column_info_tot

,Variable,Member,Type,Len,Label,Format
0,STUDYID,SDTM.DM,Char,19,Study Identifier,NaN
1,DOMAIN,SDTM.DM,Char,2,Domain Abbreviation,NaN
2,USUBJID,SDTM.DM,Char,26,Unique Subject Identifier,NaN
3,SUBJID,SDTM.DM,Char,3,Subject Identifier for the Study,NaN
4,RFSTDTC,SDTM.DM,Char,10,Subject Reference Start Date/Time,NaN
5,RFENDTC,SDTM.DM,Char,10,Subject Reference End Date/Time,NaN
6,RFXSTDTC,SDTM.DM,Char,16,Date/Time of First Study Treatment,NaN
7,RFXENDTC,SDTM.DM,Char,10,Date/Time of Last Study Treatment,NaN
8,RFICDTC,SDTM.DM,Char,10,Date/Time of Informed Consent,NaN
9,RFPENDTC,SDTM.DM,Char,10,Date/Time of End of Participation,NaN


Calling dictionaries for SDTM v3.4 as in the CDISC Guidelines:

In [43]:
df_dm = pd.DataFrame(sdtm_34_dictionaries.dict_dm).T
df_dm = df_dm.fillna({'Origin':'Collected'})
df_dm.reset_index(inplace=True)
df_dm['Variable']=df_dm['index']
df_dm

,index,Origin,Source,Core,Variable Type,Variable Order,Variable
0,STUDYID,Protocol,Sponsor,Req,SDTM,1,STUDYID
1,DOMAIN,Assigned,Sponsor,Req,SDTM,2,DOMAIN
2,USUBJID,Derived,Sponsor,Req,SDTM,3,USUBJID
3,ACTARM,Derived,Sponsor,Exp,SDTM,27,ACTARM
4,ACTARMCD,Derived,Sponsor,Exp,SDTM,26,ACTARMCD
5,ACTARMUD,Assigned,Sponsor,Exp,SDTM,29,ACTARMUD
6,ARMNRS,Assigned,Sponsor,Exp,SDTM,28,ARMNRS
7,AGE,Collected,Investigator,Exp,SDTM,19,AGE
8,AGEU,Assigned,Sponsor,Exp,SDTM,10,AGEU
9,ARM,Assigned,Sponsor,Exp,SDTM,25,ARM


##### Merge the data into a single dataframe

In [44]:
merged_dm = pd.merge(column_info_tot[['Variable','Type','Len','Format','Label']],df_dm,on='Variable')
merged_dm = pd.merge(merged_dm,dm,on='Variable',how='outer')
merged_dm = merged_dm.sort_values(by='Variable Order')
merged_dm.drop('index', axis=1, inplace=True)
merged_dm.loc[merged_dm["Variable"] == "BRTHDTC", "Format"] = "ISO 8601 datetime"
merged_dm.loc[merged_dm["Variable"] == "RFSTDTC", "Format"] = "ISO 8601 datetime"
merged_dm.loc[merged_dm["Variable"] == "RFENDTC", "Format"] = "ISO 8601 datetime"
merged_dm.loc[merged_dm["Variable"] == "RFXSTDTC", "Format"] = "ISO 8601 datetime"
merged_dm.loc[merged_dm["Variable"] == "RFXENDTC", "Format"] = "ISO 8601 datetime"
merged_dm.loc[merged_dm["Variable"] == "RFPENDTC", "Format"] = "ISO 8601 datetime"
merged_dm.loc[merged_dm["Variable"] == "RFICDTC", "Format"] = "ISO 8601 datetime"
merged_dm.loc[merged_dm["Variable"] == "DTHDTC", "Format"] = "ISO 8601 datetime"
merged_dm.loc[merged_dm["Variable"] == "DTHFL", "Format"] = "C66742"
merged_dm.loc[merged_dm["Variable"] == "AGEU", "Format"] = "C66781"
merged_dm.loc[merged_dm["Variable"] == "SEX", "Format"] = "C66731"
merged_dm.loc[merged_dm["Variable"] == "RACE", "Format"] = "C74457"
merged_dm.loc[merged_dm["Variable"] == "ETHNIC", "Format"] = "C66790"
merged_dm.loc[merged_dm["Variable"] == "ARMNRS", "Format"] = "C142179"

merged_dm.loc[merged_dm["Variable"] == "STUDYID", "Conversion Definition"] = "Set to Clinical Study B"
merged_dm.loc[merged_dm["Variable"] == "AGEU", "Conversion Definition"] = "Set to 'YEARS'"
merged_dm.loc[merged_dm["Variable"] == "USUBJID", "Conversion Definition"] = "Concatenate STUDYID, SITEID and SUBJID by '-'"


merged_dm

,Variable,Type,Len,Format,Label,Origin,Source,Core,Variable Type,Variable Order,Conversion Definition
27,STUDYID,Char,19.0,NaN,Study Identifier,Protocol,Sponsor,Req,SDTM,1,Set to Clinical Study B
13,DOMAIN,Char,2.0,NaN,Domain Abbreviation,Assigned,Sponsor,Req,SDTM,2,DM.MODULE
29,USUBJID,Char,26.0,NaN,Unique Subject Identifier,Derived,Sponsor,Req,SDTM,3,"Concatenate STUDYID, SITEID and SUBJID by '-'"
28,SUBJID,Char,3.0,NaN,Subject Identifier for the Study,Assigned,Sponsor,Req,SDTM,4,DM.PATNO
22,RFSTDTC,Char,10.0,ISO 8601 datetime,Subject Reference Start Date/Time,Derived,Sponsor,Exp,SDTM,5,NaN
19,RFENDTC,Char,10.0,ISO 8601 datetime,Subject Reference End Date/Time,Derived,Sponsor,Exp,SDTM,6,NaN
24,RFXSTDTC,Char,16.0,ISO 8601 datetime,Date/Time of First Study Treatment,Derived,Sponsor,Exp,SDTM,7,NaN
23,RFXENDTC,Char,10.0,ISO 8601 datetime,Date/Time of Last Study Treatment,Derived,Sponsor,Exp,SDTM,8,NaN
4,AGEU,Char,5.0,C66781,Age Units,Assigned,Sponsor,Exp,SDTM,10,Set to 'YEARS'
20,RFICDTC,Char,10.0,ISO 8601 datetime,Date/Time of Informed Consent,Collected,Investigator,Exp,SDTM,11,"DM.DSSTMO_N, DM.DSSTDY, DM.DSSTYR"


In [45]:
with pd.ExcelWriter(output_spec, engine='openpyxl', mode='a',if_sheet_exists='overlay') as writer:
    merged_dm.to_excel(writer,sheet_name="DM", startrow=12, index=False)

#### Supplemental Qualifiers for DM Dataset

In [46]:
ae001, ae001_meta = prs.read_sas7bdat(folder_sdtm+"/suppdm.sas7bdat")
column_info = pd.DataFrame()
column_info['Variable'] = ae001_meta.column_names
column_info['Variable'] = column_info['Variable'].str.strip(' ')
column_info.set_index('Variable',inplace=True)
column_info['Member'] = "SDTM.SUPPDM"
column_info['Type'] = ae001_meta.readstat_variable_types
column_info['Type'] = column_info['Type'].map(mapping)
column_info['Len'] = ae001_meta.variable_storage_width
column_info['Label'] = ae001_meta.column_labels
column_info['Format'] = ae001_meta.value_labels
column_info.reset_index(inplace=True)
column_info

,Variable,Member,Type,Len,Label,Format
0,STUDYID,SDTM.SUPPDM,Char,19,Study Identifier,NaN
1,RDOMAIN,SDTM.SUPPDM,Char,2,Related Domain Abbreviation,NaN
2,USUBJID,SDTM.SUPPDM,Char,33,Unique Subject Identifier,NaN
3,IDVAR,SDTM.SUPPDM,Char,1,Identifying Variable,NaN
4,IDVARVAL,SDTM.SUPPDM,Char,1,Identifying Variable Value,NaN
5,QNAM,SDTM.SUPPDM,Char,8,Qualifier Variable Name,NaN
6,QLABEL,SDTM.SUPPDM,Char,6,Qualifier Variable Label,NaN
7,QVAL,SDTM.SUPPDM,Char,2,Data Value,NaN
8,QORIG,SDTM.SUPPDM,Char,9,Origin,NaN
9,QEVAL,SDTM.SUPPDM,Char,12,Evaluator,NaN


In [47]:
merged_suppdm = pd.merge(column_info[['Variable','Type','Len','Format','Label']],df_supp,on='Variable')
merged_suppdm = merged_suppdm.sort_values(by='Variable Order')
#merged_suppdm.loc[merged_suppdm['Variable'].str.strip(' ') == 'RDOMAIN', 'Conversion Definition'] = 'DM'
merged_suppdm.loc[merged_suppdm['Variable'].str.strip(' ') == 'IDVAR', 'Conversion Definition'] = 'SDTM.DM USUBJID'
merged_suppdm.loc[merged_suppdm["Variable"] == "USUBJID", "Conversion Definition"] = "Concatenate STUDYID, SITEID and SUBJID by '-'"
merged_suppdm.loc[merged_suppdm["Variable"] == "RDOMAIN", "Conversion Definition"] = "Set to DM"
merged_suppdm.loc[merged_suppdm["Variable"] == "STUDYID", "Conversion Definition"] = "Set to Clinical Study B"
merged_suppdm.drop('index', axis=1, inplace=True)
merged_suppdm

,Variable,Type,Len,Format,Label,Origin,Source,Core,Variable Type,Variable Order,Conversion Definition
0,STUDYID,Char,19,NaN,Study Identifier,Protocol,Sponsor,Req,SDTM,1,Set to Clinical Study B
1,RDOMAIN,Char,2,NaN,Related Domain Abbreviation,Derived,Sponsor,Req,SDTM,2,Set to DM
2,USUBJID,Char,33,NaN,Unique Subject Identifier,Derived,Sponsor,Req,SDTM,3,"Concatenate STUDYID, SITEID and SUBJID by '-'"
3,IDVAR,Char,1,NaN,Identifying Variable,Assigned,Sponsor,Exp,SDTM,4,SDTM.DM USUBJID
4,IDVARVAL,Char,1,NaN,Identifying Variable Value,Derived,Sponsor,Exp,SDTM,5,NaN
5,QNAM,Char,8,NaN,Qualifier Variable Name,Assigned,Sponsor,Req,SDTM,6,NaN
6,QLABEL,Char,6,NaN,Qualifier Variable Label,Assigned,Sponsor,Req,SDTM,7,NaN
7,QVAL,Char,2,NaN,Data Value,Collected,Investigator,Req,SDTM,8,NaN
8,QORIG,Char,9,NaN,Origin,Assigned,Sponsor,Req,SDTM,9,NaN
9,QEVAL,Char,12,NaN,Evaluator,Assigned,Sponsor,Exp,SDTM,10,NaN


In [48]:
with pd.ExcelWriter(output_spec, engine='openpyxl', mode='a',if_sheet_exists='overlay') as writer:
    merged_suppdm.to_excel(writer,sheet_name="SUPPDM", startrow=12, index=False)